# [LAB] MCP 서버와 클라이언트 실습

MCP 서버를 직접 만들어 터미널에 띄우고, 노트북의 Agent가 그 서버에 연결해 도구를 쓰게 합니다.

- **2장 MCP 서버 만들고 띄우기** (예제): 셀로 서버 파일을 쓰고, 터미널에서 서버를 띄웁니다. 도구 설명 한 줄을 바꿔 보고, AI에게 서버 구조를 묻습니다.
- **3장 클라이언트로 연결하기** (과제): 서버에 연결해 도구를 가져오는 두 곳과, 그 도구로 Agent 그래프를 만드는 네 곳을 AI와 함께 채웁니다. 마지막에 연결 방식을 SSE에서 stdio로 한 줄 바꿔 비교합니다.

셀은 위에서부터 차례로 실행합니다. 3장은 2장의 서버가 터미널에 켜져 있어야 동작합니다.

**목차**

- 1. 실습 방법
- 2. MCP 서버 만들고 띄우기 (예제)
  - 2.1 서버 만들기 (예제)
  - 2.2 Tool 정의하기 (예제)
  - 2.3 Resource 정의하기 (예제)
  - 2.4 Prompt 정의하기 (예제)
  - 2.5 실행 방식 정하기 (예제)
  - 2.6 AI에게 물어보기
- 3. 클라이언트로 연결하기 (예제) (과제)
  - 3.1 연결 방식 고르기 (예제)
  - 3.2 클라이언트 만들기 (과제)
  - 3.3 도구 목록 가져오기 (과제)
  - 3.4 LLM과 도구 노드에 도구 연결하기 (과제)
  - 3.5 agent 노드 (과제)
  - 3.6 그래프 연결과 컴파일 (과제)
  - 3.7 질문하기 (예제)
  - 3.8 AI에게 물어보기
- 4. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 이 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 이 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

"정답 보기"의 코드는 빈칸 줄과 같은 들여쓰기로 보여 줍니다. 붙여 넣을 때는 빈칸 줄 전체(줄 앞 공백까지)를 지우고 그 자리에 붙여 넣습니다.

**막혔을 때: 스스로 먼저 확인합니다**

1. 오류가 난 셀과 오류의 마지막 줄을 AI에게 붙여 넣고, 원인과 고칠 곳을 물어봅니다. 처음 한 번은 "저장소 맨 위 AGENTS.md를 읽고 도와줘"라고 먼저 말합니다. AI가 설치·경로·연결 문제의 해결 순서를 알게 됩니다.
2. 설정 문제로 보이면 맨 위 환경 점검 셀을 다시 실행하고, ⛔ 줄의 → 안내대로 고칩니다.
3. 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다.
4. 옆 사람 화면과 비교해 봅니다.
5. 그래도 안 되면 오류 화면을 그대로 두고 강사에게 보여 줍니다. 그동안 그 셀은 건너뛰고 다음 절을 진행해도 됩니다.

## 2. MCP 서버 만들고 띄우기 <small>(예제)</small>

FastMCP로 MCP 서버를 만듭니다. 서버는 노트북 셀 안에서 돌리지 않고, **셀이 서버 파일(`mcp_server.py`)을 쓰고 터미널에서 그 파일을 실행**합니다. 서버는 켜 둔 동안 계속 요청을 기다리기 때문에, 노트북 셀에서 돌리면 다른 셀을 실행할 수 없습니다.

아래 다섯 셀이 한 파일을 나눠 씁니다. 첫 셀은 파일을 새로 쓰고(`%%writefile`), 나머지는 이어 씁니다(`%%writefile -a`). 그래서 한 셀을 고쳤다면 첫 셀부터 다시 실행해 파일을 새로 씁니다.

서버에는 3대 Primitive가 모두 들어 있습니다.

- **Tool 5개:** 현재 시간, BMI 계산, 할 일 추가·목록·완료
- **Resource 1개:** 서버 정보(`config://app-info`)
- **Prompt 1개:** 할 일 요약 요청 템플릿

다섯 셀을 차례로 실행한 뒤, 아래 안내대로 터미널에서 서버를 띄웁니다.

### 2.1 서버 만들기 <small>(예제)</small>

In [ ]:
%%writefile mcp_server.py
"""
MCP 서버 만들기 (FastMCP)

터미널에서 실행한다. 기본은 HTTP/SSE 모드(포트 8000), --stdio를 붙이면 stdio 모드로 실행한다.
"""

import sys
from datetime import datetime

from fastmcp import FastMCP

# FastMCP: Anthropic MCP SDK 위에서 동작하는 고수준 래퍼.
# @mcp.tool(), @mcp.resource(), @mcp.prompt() 데코레이터로 기능을 정의한다.
mcp = FastMCP("AI 강의 튜토리얼 서버")

**성공 기준**

- 🟢 셀 아래에 `Writing mcp_server.py` 또는 `Overwriting mcp_server.py`가 출력되었다

### 2.2 Tool 정의하기 <small>(예제)</small>

**✍️ CHANGE_HERE: 도구 설명을 바꿔 봅니다**

먼저 그대로 두고 3장의 도구 목록까지 실행해 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 아래 따옴표 안 설명을 바꾸고(예: "지금 한국 날짜와 시각을 알려 준다."), 2.1 서버 만들기 셀부터 2.5 실행 방식 정하기 셀까지 다섯 셀을 차례로 다시 실행합니다. 이 셀만 다시 실행하면 파일 맨 끝(서버 실행 줄 뒤)에 붙어 반영되지 않습니다. 터미널에서 **Ctrl+C**로 서버를 끄고 같은 명령으로 다시 띄운 뒤, 3장의 연결 셀과 도구 목록 셀을 다시 실행합니다. 서버의 도구 설명은 클라이언트가 가져가 LLM이 도구를 고르는 근거로 씁니다.

In [ ]:
%%writefile -a mcp_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── Tool: LLM이 호출할 수 있는 함수 ──────────────────────────────────────────
@mcp.tool()
def get_current_time() -> str:
    # ✍️ CHANGE_HERE: 도구 설명을 바꿔 봅니다
    """현재 날짜와 시간을 반환한다."""
    return datetime.now().strftime("%Y년 %m월 %d일 %H:%M:%S")


@mcp.tool()
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """체중(kg)과 키(cm)로 BMI를 계산한다."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m**2)
    if bmi < 18.5:
        category = "저체중"
    elif bmi < 23:
        category = "정상"
    elif bmi < 25:
        category = "과체중"
    else:
        category = "비만"
    return {"bmi": round(bmi, 1), "category": category}


_tasks: list[dict] = []


@mcp.tool()
def add_task(title: str, priority: int = 1) -> str:
    """할 일 목록에 새 항목을 추가한다. priority는 1(낮음)~3(높음)."""
    task = {"id": len(_tasks) + 1, "title": title, "priority": priority, "done": False}
    _tasks.append(task)
    return f"할 일 추가됨: [{task['id']}] {title} (우선순위: {priority})"


@mcp.tool()
def list_tasks() -> list[dict]:
    """현재 할 일 목록을 반환한다."""
    return _tasks if _tasks else [{"message": "할 일이 없습니다."}]


@mcp.tool()
def complete_task(task_id: int) -> str:
    """할 일을 완료 처리한다."""
    for task in _tasks:
        if task["id"] == task_id:
            task["done"] = True
            return f"완료: {task['title']}"
    return f"ID {task_id}를 찾을 수 없습니다."

**성공 기준**

- 🟢 셀 아래에 `Appending to mcp_server.py`가 출력되었다
- 🟢 (설명을 바꾼 뒤) 서버를 다시 띄우고 3장 도구 목록을 다시 출력하면 `get_current_time` 옆 설명이 바꾼 글로 보인다
- ⛔ 설명이 그대로다 → 2.1 셀부터 다섯 셀을 다시 실행했는지, 서버를 Ctrl+C로 끄고 다시 띄웠는지, 3장 도구 목록 셀을 다시 실행했는지 봅니다

### 2.3 Resource 정의하기 <small>(예제)</small>

In [ ]:
%%writefile -a mcp_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── Resource: LLM에게 제공할 데이터/문서 ──────────────────────────────────────
@mcp.resource("config://app-info")
def get_app_info() -> str:
    """이 MCP 서버의 기본 정보를 제공한다."""
    return """
    서버명: AI 강의 튜토리얼 MCP 서버
    버전: 1.0.0
    기능: 시간 조회, BMI 계산, 할 일 관리
    """

### 2.4 Prompt 정의하기 <small>(예제)</small>

In [ ]:
%%writefile -a mcp_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── Prompt: 자주 쓰는 프롬프트 템플릿 ────────────────────────────────────────
@mcp.prompt()
def task_summary_prompt(user_name: str) -> str:
    """할 일 목록 요약 요청 프롬프트를 생성한다."""
    return f"{user_name}님의 할 일 목록을 우선순위별로 정리하고 오늘 집중해야 할 항목을 추천해줘."

### 2.5 실행 방식 정하기 <small>(예제)</small>

In [ ]:
%%writefile -a mcp_server.py
# 이어 쓰기 셀: 고쳤다면 2.1 셀부터 차례로 다시 실행합니다(이 셀만 다시 실행하면 반영되지 않습니다)
# ── 실행 ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    if "--stdio" in sys.argv:
        # stdio 모드: Claude Code, Cursor 등 MCP Host와 직접 연결
        mcp.run(transport="stdio")
    else:
        print("MCP 서버 시작: http://localhost:8000/sse")
        print("종료하려면 Ctrl+C")
        # HTTP/SSE 모드: 클라이언트가 localhost:8000으로 연결
        mcp.run(transport="sse", host="0.0.0.0", port=8000)

위 다섯 셀을 차례로 실행하면 이 폴더에 서버 파일이 만들어집니다. 서버는 터미널에서 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/mcp_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다.
4. 서버 파일을 고쳤다면 터미널에서 **Ctrl+C**로 끄고, 2.1 셀부터 2.5 셀까지 차례로 모두 다시 실행한 뒤 같은 명령으로 다시 띄웁니다. 고친 셀 하나만 다시 실행하면 파일 끝에 덧붙기만 해서 반영되지 않습니다.

**성공 기준** (터미널 화면)

- 🟢 `MCP 서버 시작: http://localhost:8000/sse`와 `종료하려면 Ctrl+C` 두 줄이 보인다
- 🟢 그 아래 FastMCP 안내 상자(서버 이름 `AI 강의 튜토리얼 서버`가 적힌 상자)가 나오고, 맨 끝 줄에 `Uvicorn running on`과 `(Press CTRL+C to quit)`가 보인다
- 🟢 명령이 끝나지 않고 기다린다(새 명령을 입력하는 줄이 돌아오지 않는다)
- ⛔ 명령이 곧바로 끝나고 오류가 나온다 → 오류 마지막 줄을 AI에게 보여 줍니다
- ⛔ 오류에 `address already in use` 또는 `10048`이 보인다 → 이미 켜 둔 서버가 있습니다. 앞서 서버를 띄운 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

**볼 지점**
- 서버를 띄운 터미널은 3장이 끝날 때까지 켜 둡니다. 3장에서 도구를 호출할 때마다 이 터미널에 요청 기록(`POST /messages/...` 같은 줄)이 늘어납니다.
- 안내 상자에 새 버전 안내(`Update available`)가 함께 나올 수 있습니다. 실습과 관계없으니 그대로 둡니다.

### 2.6 AI에게 물어보기

**프롬프트 힌트** 위 다섯 셀의 서버 코드를 붙여 넣고, `@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`가 각각 무엇을 등록하는지와 클라이언트가 도구 목록을 가져오면 몇 개가 보일지 물어봅니다. 코드는 고치지 말고 설명만 해 달라고 합니다.

**성공 기준**

- 🟢 도구는 5개(`get_current_time`, `calculate_bmi`, `add_task`, `list_tasks`, `complete_task`)라고 답한다
- 🟢 Resource(`config://app-info`)와 Prompt(`task_summary_prompt`)는 도구 목록에 들어가지 않는다는 점을 짚는다
- ⛔ 도구가 7개라고 답한다 → Resource와 Prompt도 도구로 가져오는지 다시 묻습니다
- ⛔ 코드를 고쳐 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다

## 3. 클라이언트로 연결하기 <small>(예제) (과제)</small>

노트북에서 Agent(Host)를 만들고, 그 안의 Client로 2장의 서버에 연결합니다. 서버의 도구를 가져와 LLM에 넘기고, 과정 1에서 만든 것과 같은 모양의 LangGraph 그래프(agent ⇄ tools)로 질문에 답하게 합니다.

빈칸은 여섯 곳입니다.

- **연결:** 클라이언트 만들기, 도구 목록 가져오기
- **Agent 그래프:** LLM과 도구 노드에 도구 연결하기, agent 노드, 조건부 엣지, 컴파일

서버와 주고받는 코드 앞에는 `await`가 붙습니다. 응답이 올 때까지 기다린다는 표시입니다. 노트북 셀에서는 `await`를 바로 쓸 수 있습니다.

마지막 질문 셀은 빈칸 여섯 곳을 **모두 채우고 실행한 뒤에** 동작합니다.

**왜 배우나:** 도구를 서버에서 가져온다는 점만 다르고, 나머지는 과정 1의 `bind_tools`·ToolNode·조건부 엣지를 그대로 씁니다. 도구가 어디에 있든 Agent 쪽 코드는 같다는 것을 직접 확인합니다.

준비 코드입니다. 필요한 도구를 불러오고 설정 파일을 읽습니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
MCP 클라이언트 실습: 터미널에 켜 둔 MCP 서버에 연결해 LangGraph Agent가 도구로 사용한다.
"""

import sys
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv();

### 3.1 연결 방식 고르기 <small>(예제)</small>

**✍️ CHANGE_HERE: 연결 방식을 "sse"에서 "stdio"로 바꿔 봅니다**

처음에는 그대로(`"sse"`) 두고 3장 끝까지 실행합니다. 모두 동작하면 이 줄을 `"stdio"`로 바꾸고 이 셀부터 질문하기 셀까지 차례로 다시 실행합니다. SSE는 터미널에 켜 둔 서버에 주소로 접속하고, stdio는 노트북이 서버 파일을 직접 실행해 주고받습니다. 비교가 끝나면 `"sse"`로 되돌립니다.

In [ ]:
# ── MCP 서버 연결 설정 ────────────────────────────────────────────────────────
# SSE(Server-Sent Events): HTTP 기반 MCP 연결 방식.
# 서버를 HTTP 모드로 실행하면 /sse 엔드포인트가 열린다. 터미널에 켜 둔 서버에 주소로 접속한다.
# stdio: 노트북이 서버 파일을 직접 실행해 표준 입출력으로 주고받는다.
CONNECTIONS = {
    "sse": {"transport": "sse", "url": "http://localhost:8000/sse"},
    "stdio": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["mcp_server.py", "--stdio"],
    },
}
# ✍️ CHANGE_HERE: 연결 방식을 "sse"에서 "stdio"로 바꿔 봅니다
TRANSPORT = "sse"
MCP_CONFIG = {"tutorial": CONNECTIONS[TRANSPORT]}

**성공 기준**

- 🟢 셀이 오류 없이 끝난다(설정만 만들어 출력은 없다)
- 🟢 ("stdio"로 바꾸고 다시 실행한 뒤) 서버 터미널을 Ctrl+C로 꺼 두어도 도구 목록에 같은 도구 5개가 나온다
- 🟢 ("stdio"로 바꾸고 다시 실행한 뒤) BMI 답은 SSE 때와 같지만, 할 일 질문에는 추가했는데 목록이 비어 있다는 식으로 답한다
- ⛔ ("stdio"로 바꾼 뒤) 오류에 `FileNotFoundError`나 `NotImplementedError`가 보인다 → 이 PC에서 stdio 방식이 서버 프로그램을 띄우지 못한 것입니다. 오류를 AI에게 보여 원인을 물어보고, 풀리지 않으면 `"sse"`로 되돌려 넘어갑니다

### 3.2 클라이언트 만들기 <small>(과제)</small>

**✍️ WRITE_HERE: MCP 설정으로 MultiServerMCPClient를 생성해 주세요.**

위 셀에서 만든 연결 설정 `MCP_CONFIG`로 클라이언트를 만들어 변수 `client`에 담는 한 줄입니다. 이 줄은 서버에 바로 접속하지 않습니다. 설정만 들고 있다가 다음 셀에서 도구를 가져올 때 접속합니다.

**프롬프트 힌트** `langchain_mcp_adapters`의 `MultiServerMCPClient`에 위 셀의 `MCP_CONFIG`를 넘겨 변수 `client`를 만드는 한 줄을 요청합니다. 불러오기는 이미 되어 있으니 다시 쓰지 말라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 끝난다(클라이언트만 만들어 출력은 없다)
- ⛔ `NameError: name 'MCP_CONFIG' is not defined`가 보인다 → 위 연결 방식 셀을 먼저 실행합니다

In [ ]:
# ✍️ WRITE_HERE: MCP 설정으로 MultiServerMCPClient를 생성해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
client = MultiServerMCPClient(MCP_CONFIG)
```

</details>

### 3.3 도구 목록 가져오기 <small>(과제)</small>

**✍️ WRITE_HERE: MCP 서버에서 도구 목록을 가져와 주세요.**

서버에 접속해 도구 이름과 설명을 읽어 LangChain Tool 목록으로 받아 `tools`에 담는 한 줄입니다. 서버와 주고받는 동안 기다려야 하므로 `await`가 필요합니다. 아래 줄들이 받은 도구를 하나씩 출력합니다.

**프롬프트 힌트** 위에서 만든 `client`로 서버의 도구 목록을 가져와 변수 `tools`에 담는 한 줄을 요청합니다. 노트북 셀에서 바로 `await`를 쓸 수 있으니 `asyncio.run`이나 새 함수로 감싸지 말라고 적습니다.

**성공 기준**

- 🟢 `=== 사용 가능한 MCP 도구 ===` 아래에 도구 5줄이 나온다(`- get_current_time: 현재 날짜와 시간을 반환한다.`로 시작)
- ⛔ 오류에 `ConnectError`나 `Connection refused`가 보인다 → 2장의 서버 터미널이 켜져 있는지 봅니다
- ⛔ `'coroutine' object is not iterable`이 보인다 → `await`가 빠졌습니다
- ⛔ `asyncio.run() cannot be called from a running event loop`가 보인다 → `asyncio.run` 없이 `await`만 쓰게 다시 요청합니다

In [ ]:
# get_tools()는 서버에 연결해 도구 목록을 가져온다.
# ✍️ WRITE_HERE: MCP 서버에서 도구 목록을 가져와 주세요.
...  # ✍️ 여기를 채웁니다
print("=== 사용 가능한 MCP 도구 ===")
for t in tools:
    print(f"  - {t.name}: {t.description}")
print()

<details><summary>정답 보기</summary>

```python
tools = await client.get_tools()
```

</details>

### 3.4 LLM과 도구 노드에 도구 연결하기 <small>(과제)</small>

**✍️ WRITE_HERE: 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.**

두 줄입니다. 하나는 `get_llm()`으로 만든 모델에 서버의 도구 목록을 알려 주어 `llm`에 담고, 다른 하나는 같은 도구 목록으로 도구를 실제로 실행하는 노드 `tool_node`를 만듭니다. 과정 1의 Tool Calling과 LangGraph에서 쓴 방법과 같습니다.

**프롬프트 힌트** 앞 셀에서 받은 `tools`를 `get_llm()`으로 만든 모델에 연결해 `llm`에, LangGraph의 `ToolNode`로 감싸 `tool_node`에 담는 두 줄을 요청합니다. 도구를 새로 정의하지 말고 받은 `tools`만 쓰라고 적습니다.

**성공 기준**

- 🟢 질문하기 셀에서 시간·BMI 질문의 답에 실제 시각과 BMI 숫자가 들어 있다
- ⛔ 질문하기 셀에서 "현재 시간을 알 수 없다"는 식으로 답한다 → 모델에 도구를 연결했는지(`bind_tools`) 다시 봅니다

In [ ]:
# ✍️ WRITE_HERE: 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
llm = get_llm().bind_tools(tools)
tool_node = ToolNode(tools)
```

</details>

### 3.5 agent 노드 <small>(과제)</small>

**✍️ WRITE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.**

`agent_node`는 Agent가 한 번 생각하는 단계입니다. 상태에 쌓인 메시지를 도구가 연결된 `llm`에 보내고, 받은 응답을 `messages`에 더하도록 딕셔너리로 돌려주는 한 줄입니다.

**프롬프트 힌트** `agent_node` 함수 안에서 `state`의 메시지 목록을 앞 셀의 `llm`에 보내고, 응답을 `messages` 키의 리스트에 담은 딕셔너리로 돌려주는 한 줄을 요청합니다. 함수 이름과 인자는 바꾸지 말라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 끝난다(함수만 정의하므로 출력은 없다)
- ⛔ 질문하기 셀에서 `A:` 뒤에 질문이 그대로 다시 나온다 → 줄 앞에 `return`이 있는지, 응답을 `{"messages": [...]}` 딕셔너리에 담아 돌려주는지 봅니다
- ⛔ 질문하기 셀에서 `InvalidUpdateError`가 보인다 → 응답을 `{"messages": [...]}` 딕셔너리에 담아 돌려주는지 봅니다

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    # ✍️ WRITE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.
    ...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
    return {"messages": [llm.invoke(state["messages"])]}
```

</details>

### 3.6 그래프 연결과 컴파일 <small>(과제)</small>

**✍️ WRITE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

두 줄입니다. agent 노드 다음에 LLM이 도구 호출을 요청했으면 tools로, 아니면 끝(END)으로 가게 조건부 엣지를 답니다. 그리고 tools 노드가 도구를 실행한 뒤에는 다시 agent로 돌아와 결과를 보고 답하게 합니다.

**프롬프트 힌트** agent 노드 뒤에 LangGraph의 `tools_condition`으로 조건부 엣지를 달고, 노드 `tools`에서 `agent`로 돌아오는 엣지를 더하는 두 줄을 요청합니다. 노드 이름은 위에 등록한 `agent`, `tools`를 그대로 쓰라고 적습니다.

**성공 기준**

- 🟢 질문하기 셀에서 할 일 질문의 답에 "LangGraph 공부"(우선순위 3)가 든 목록이 나온다
- ⛔ 질문하기 셀에서 `A:` 뒤에 문장 대신 `[{'type': 'text', 'text': …}]` 모양의 도구 결과가 그대로 나온다 → tools에서 agent로 돌아오는 엣지를 봅니다
- ⛔ 질문하기 셀에서 `A:`가 비어 있거나 도구를 부르겠다는 말로 끝난다 → `tools_condition` 조건부 엣지 줄을 봅니다

**✍️ WRITE_HERE: 그래프를 컴파일해 주세요.**

노드와 엣지를 다 붙인 `builder`를 실행할 수 있는 그래프로 만들어 `app`에 담는 한 줄입니다. 다음 셀이 `app.ainvoke(...)`로 이 그래프를 실행합니다.

**프롬프트 힌트** 위 `builder`를 컴파일해 변수 `app`에 담는 한 줄을 요청합니다. 체크포인터 같은 옵션은 넣지 말라고 적습니다.

**성공 기준**

- 🟢 질문하기 셀이 오류 없이 세 질문에 모두 답한다
- ⛔ `ValueError`와 함께 노드 이름이 보인다 → 조건부 엣지 빈칸의 노드 이름을 봅니다

In [ ]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
# ✍️ WRITE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.
...  # ✍️ 여기를 채웁니다

# ✍️ WRITE_HERE: 그래프를 컴파일해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
```

```python
app = builder.compile()
```

</details>

### 3.7 질문하기 <small>(예제)</small>

**✍️ CHANGE_HERE: 첫 질문을 도구가 필요 없는 질문으로 바꿔 봅니다**

빈칸을 모두 채워 세 질문의 답을 본 뒤에 합니다. `# ✍️ CHANGE_HERE:` 아래 따옴표 안 질문을 도구가 필요 없는 질문(예: "MCP가 뭐야?")으로 바꿔 이 셀을 다시 실행합니다. LLM이 도구를 호출하지 않기로 판단하면 조건부 엣지가 tools를 거치지 않고 바로 끝냅니다.

In [ ]:
# 도구를 사용해 요청 처리
questions = [
    # ✍️ CHANGE_HERE: 첫 질문을 도구가 필요 없는 질문으로 바꿔 봅니다
    "지금 몇 시야?",
    "체중 70kg, 키 175cm면 BMI가 얼마야?",
    '할 일 목록에 "LangGraph 공부" (우선순위 3)를 추가하고 현재 목록을 보여줘.',
]

for question in questions:
    print(f"Q: {question}")
    result = await app.ainvoke(
        {
            "messages": [HumanMessage(content=question)],
        }
    )
    print(f"A: {result['messages'][-1].content}")
    print()

**성공 기준**

- 🟢 (바꾼 뒤) 도구가 필요 없는 질문에는 시각 같은 도구 결과 없이 LLM이 바로 답한다
- ⛔ (바꾼 뒤) 셀이 끝나지 않고 계속 돈다 → 셀 실행을 중지(■)하고 조건부 엣지 빈칸을 다시 봅니다

**볼 지점** (질문하기 셀)
- `Q:` 줄 아래 `A:` 줄에 답이 나옵니다. 세 질문 모두 LLM이 서버의 도구를 호출해 얻은 값으로 답합니다.
  - 시간 질문: 지금 날짜와 시각
  - BMI 질문: BMI 22.9, 정상 범위
  - 할 일 질문: "LangGraph 공부"(우선순위 3)가 든 목록
- 서버 터미널에 도구 호출마다 요청 기록이 늘어납니다. Agent는 노트북에서, 도구는 터미널의 서버에서 실행된다는 뜻입니다.
- 서버를 켠 채로 질문하기 셀을 다시 실행하면 할 일 목록에 "LangGraph 공부"가 하나씩 더 쌓입니다. 서버가 켜져 있는 동안 목록을 기억하기 때문이며 정상입니다. 서버를 껐다 켜면 비워집니다.
- 답의 말투와 모양은 실행할 때마다 조금씩 다릅니다.

**stdio로 바꿔 보기**

3.1 연결 방식 고르기 셀의 `TRANSPORT`를 `"stdio"`로 바꾸고 질문하기 셀까지 다시 실행해 결과를 SSE 때와 비교합니다(성공 기준은 3.1 셀 아래에 있습니다). stdio에서는 노트북이 도구를 호출할 때마다 서버 프로그램을 새로 실행하고 끝냅니다. 그래서 할 일을 추가한 서버와 목록을 보여 주는 서버가 서로 다른 서버가 되어, 추가한 할 일이 목록에 보이지 않습니다.

### 3.8 AI에게 물어보기

**프롬프트 힌트** 3.1 연결 방식 고르기 셀의 연결 설정을 붙여 넣고, SSE로 연결했을 때는 할 일 목록이 남는데 stdio로 바꾸면 추가한 할 일이 목록에 보이지 않는 이유를 물어봅니다. 코드는 고치지 말고 두 연결 방식에서 서버가 언제 켜지고 꺼지는지로 설명해 달라고 합니다.

**성공 기준**

- 🟢 SSE는 터미널에 켜 둔 서버 하나가 계속 떠 있어 할 일 목록을 기억한다고 설명한다
- 🟢 stdio는 도구를 호출할 때마다 서버 프로그램이 새로 실행되고 끝나서, 추가한 서버와 목록을 보여 주는 서버가 다르다고 설명한다
- ⛔ "LLM이 목록을 잊어서"라고만 답한다 → 할 일 목록이 LLM이 아니라 서버 프로그램 안에 있다는 점을 알려 주고 다시 묻습니다
- ⛔ 코드를 고쳐 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다

## 4. 정리

**이 노트북에서 한 것**
- FastMCP로 Tool·Resource·Prompt가 든 MCP 서버를 셀로 쓰고, 터미널에서 HTTP/SSE 모드로 띄웠습니다.
- `MultiServerMCPClient`로 서버에 연결하고 `get_tools`로 도구 5개를 가져왔습니다. Resource와 Prompt는 도구 목록에 들어오지 않습니다.
- 가져온 도구를 `bind_tools`와 ToolNode에 넘겨, 과정 1과 같은 agent ⇄ tools 그래프로 질문에 답하게 했습니다.
- 연결 방식을 stdio로 바꾸면 노트북이 서버를 직접 실행합니다. 호출할 때마다 서버가 새로 떠서 할 일 목록이 이어지지 않았습니다.

실습이 끝나면 서버를 띄운 터미널에서 **Ctrl+C**로 서버를 끕니다.